# Notebook 1 · Introduction to Linear Models

Learn what a linear model is, how to measure its error, and how a computer finds the best line (closed form and gradient descent), using pandas, numpy and scikit-learn.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Rathachai/DA-LAB/blob/gh-pages/learnings/linear/notebooks/nb-01-linear-intro.ipynb)

- Lecture: [en-01-linear-intro.md](https://github.com/Rathachai/DA-LAB/blob/gh-pages/learnings/linear/en-01-linear-intro.md)
- Interactive lab(s): [LM101](https://rathachai.github.io/DA-LAB/learnings/linear/lm101.html)

Powered by: Sonnet  |  AI Whisperer: rathachai.github.io

## 1. Setup
We use pandas for tables, numpy for arithmetic, scikit-learn for models and metrics, and matplotlib only for plots.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (mean_absolute_error, mean_squared_error,
                             r2_score, mean_absolute_percentage_error)
pd.options.display.float_format = '{:.3f}'.format
print('Libraries ready')

## 2. A tiny data table: `id`, `X`, `y`
Every row is one sample. `id` only names the row, `x` (load) is the input we know, `y` (temperature) is what we want to predict.

In [ ]:
motor = pd.DataFrame({'id': [1, 2, 3, 4, 5],
                      'load_kW': [10, 20, 30, 40, 50],
                      'temp_C': [52, 61, 68, 80, 88]})
motor

Plot it: the five points rise roughly along a straight line.

In [ ]:
plt.scatter(motor['load_kW'], motor['temp_C'], color='tab:blue')
plt.title('Motor winding temperature vs load')
plt.xlabel('Load x (kW)'); plt.ylabel('Temperature y (°C)')
plt.grid(alpha=0.3); plt.show()

## 3. A bigger data set: `lm101_points.csv`
We load it straight from GitHub Pages and look at the first rows.

In [ ]:
url = "https://rathachai.github.io/DA-LAB/learnings/linear/data/lm101_points.csv"
df = pd.read_csv(url)
df.head()

`describe()` gives count, mean, spread and range of every column.

In [ ]:
df.describe()

Scatter plot of the 40 points.

In [ ]:
plt.scatter(df['x'], df['y'], color='tab:blue')
plt.title('LM101 data: y vs x')
plt.xlabel('x (input units)'); plt.ylabel('y (output units)')
plt.grid(alpha=0.3); plt.show()

## 4. A manual line and its residuals
Choose a slope `m` and intercept `c` by eye. The prediction is `y_hat = m*x + c`; the **residual** is `y - y_hat` (measured minus predicted). Both become new columns.

In [ ]:
m, c = 1.0, 4.0            # our guess, tweak it!
df['y_hat'] = m * df['x'] + c
df['residual'] = df['y'] - df['y_hat']
df.head()

Draw the line and a vertical segment for every residual.

In [ ]:
plt.scatter(df['x'], df['y'], label='measured y')
xs = np.sort(df['x'])
plt.plot(xs, m * xs + c, color='tab:red', label=f'line: y = {m}x + {c}')
plt.vlines(df['x'], df['y_hat'], df['y'], color='gray', alpha=0.6, label='residuals')
plt.title('Manual line and residuals'); plt.xlabel('x (input units)'); plt.ylabel('y (output units)')
plt.legend(); plt.show()

## 5. Error metrics
First by hand with numpy, then with scikit-learn. They must agree.

- MAE: average absolute miss. MSE: average squared miss. RMSE: square root of MSE (same unit as y).
- MAPE: average miss as a percentage of y. R²: share of the variation in y explained (1 = perfect, 0 = no better than the average).

In [ ]:
e = df['residual'].to_numpy(); y = df['y'].to_numpy()
by_hand = {'MAE': np.mean(np.abs(e)),
           'MSE': np.mean(e**2),
           'RMSE': np.sqrt(np.mean(e**2)),
           'MAPE (%)': 100 * np.mean(np.abs(e / y)),
           'R2': 1 - np.sum(e**2) / np.sum((y - y.mean())**2)}
yh = df['y_hat']
mse = mean_squared_error(y, yh)
by_sklearn = {'MAE': mean_absolute_error(y, yh), 'MSE': mse, 'RMSE': np.sqrt(mse),
              'MAPE (%)': 100 * mean_absolute_percentage_error(y, yh), 'R2': r2_score(y, yh)}
pd.DataFrame({'by hand (numpy)': by_hand, 'scikit-learn': by_sklearn})

## 6. Closed-form least squares (OLS)
The best line has an exact formula: `m = Sxy / Sxx` and `c = ybar - m * xbar`. Here `Sxy = sum((x-xbar)(y-ybar))` and `Sxx = sum((x-xbar)^2)`. We compare it with scikit-learn's `LinearRegression`.

In [ ]:
x = df['x']
Sxy = ((x - x.mean()) * (df['y'] - df['y'].mean())).sum()
Sxx = ((x - x.mean())**2).sum()
m_ols = Sxy / Sxx
c_ols = df['y'].mean() - m_ols * x.mean()
lr = LinearRegression().fit(df[['x']], df['y'])
pd.DataFrame({'slope m': [m, m_ols, lr.coef_[0]], 'intercept c': [c, c_ols, lr.intercept_]},
             index=['manual guess', 'OLS formula', 'sklearn LinearRegression'])

Plot the manual line and the OLS line together, and compare the RMSE of both.

In [ ]:
plt.scatter(df['x'], df['y'], label='data')
plt.plot(xs, m * xs + c, '--', color='tab:orange', label='manual line')
plt.plot(xs, m_ols * xs + c_ols, color='tab:green', label='OLS line')
plt.title('Manual line vs least-squares line'); plt.xlabel('x (input units)'); plt.ylabel('y (output units)')
plt.legend(); plt.show()
rm = lambda p: np.sqrt(mean_squared_error(df['y'], p))
print('RMSE manual:', round(rm(m*xs*0 + df['y_hat']), 3), '| RMSE OLS:', round(rm(lr.predict(df[['x']])), 3))

## 7. Gradient descent from scratch
Instead of a formula we walk downhill on the loss (MSE). Each step: predict, compute the residuals, then move `m` and `c` against the gradient: `g_m = -2*mean(x*e)`, `g_c = -2*mean(e)`. This is the one place where we write the algorithm ourselves. We keep a **history** of every step.

In [ ]:
def gradient_descent(x, y, lr=0.01, steps=200, m0=0.0, c0=0.0):
    m, c, hist = m0, c0, []
    for i in range(steps):
        e = y - (m * x + c)
        hist.append((i, m, c, np.mean(e**2)))
        m -= lr * (-2 * np.mean(x * e))
        c -= lr * (-2 * np.mean(e))
    return m, c, pd.DataFrame(hist, columns=['step', 'm', 'c', 'mse'])

xv, yv = df['x'].to_numpy(), df['y'].to_numpy()
m_gd, c_gd, hist = gradient_descent(xv, yv, lr=0.01, steps=1000)
print(f'GD: m={m_gd:.3f}, c={c_gd:.3f}  |  sklearn: m={lr.coef_[0]:.3f}, c={lr.intercept_:.3f}')
hist.head()

The loss curve should fall quickly and then flatten out.

In [ ]:
plt.plot(hist['step'], hist['mse'])
plt.title('Gradient descent: loss curve (lr = 0.01)')
plt.xlabel('Iteration'); plt.ylabel('MSE (y units squared)'); plt.grid(alpha=0.3); plt.show()

Watch the line improve: the red-to-green lines are the model at steps 0, 5, 20, 100 and 999.

In [ ]:
plt.scatter(df['x'], df['y'], color='lightgray', label='data')
for k in [0, 5, 20, 100, 999]:
    r = hist.loc[k]
    plt.plot(xs, r['m'] * xs + r['c'], label=f'step {k}')
plt.title('The line at several iterations'); plt.xlabel('x (input units)'); plt.ylabel('y (output units)')
plt.legend(); plt.show()

## 8. The learning rate
Too small: converges very slowly. Good: smooth and fast. Too large: the steps overshoot and the loss **diverges**. Note the log scale on the y-axis.

In [ ]:
rates = {'too small (0.0005)': 0.0005, 'good (0.01)': 0.01, 'too large (0.1)': 0.1}
for name, a in rates.items():
    h = gradient_descent(xv, yv, lr=a, steps=100)[2]
    plt.plot(h['step'], h['mse'], label=name)
plt.yscale('log'); plt.title('Effect of the learning rate')
plt.xlabel('Iteration'); plt.ylabel('MSE (log scale)'); plt.legend(); plt.grid(alpha=0.3); plt.show()

## 9. Standardising x
Subtract the mean and divide by the standard deviation (`StandardScaler`). Now x is roughly between -2 and +2, and one learning rate works well. Here is the standardised column.

In [ ]:
sc = StandardScaler()
df['x_std'] = sc.fit_transform(df[['x']])
df[['id', 'x', 'x_std', 'y']].head()

Compare convergence with the **same** learning rate (0.05) on raw and standardised x. On raw x this rate is too large and the loss explodes; on standardised x it works well.

In [ ]:
h_raw = gradient_descent(df['x'].to_numpy(), yv, lr=0.05, steps=60)[2]
h_std = gradient_descent(df['x_std'].to_numpy(), yv, lr=0.05, steps=60)[2]
plt.plot(h_raw['step'], h_raw['mse'], label='raw x')
plt.plot(h_std['step'], h_std['mse'], label='standardised x')
plt.yscale('log'); plt.title('Same learning rate (0.05), different scaling')
plt.xlabel('Iteration'); plt.ylabel('MSE (log scale)'); plt.legend(); plt.grid(alpha=0.3); plt.show()
print('Final MSE  raw:', h_raw['mse'].iloc[-1], ' standardised:', round(h_std['mse'].iloc[-1], 4))

## 10. Summary / Key takeaways

- A table has `id` (names the row), `X` (inputs) and `y` (target to predict).
- A linear model is `y_hat = m*x + c`; residual = `y - y_hat`.
- MAE, MSE, RMSE, MAPE and R² score the error; hand-made values match `sklearn.metrics`.
- The best line comes from the closed-form OLS formula (same as `LinearRegression`) or from gradient descent.
- The learning rate matters: too small is slow, too large diverges.
- Standardising x makes gradient descent converge faster and more safely.

Powered by: Sonnet  |  AI Whisperer: rathachai.github.io